# Eulerian Video Magnification, accelerated on CUDA

**Eulerian Video Magnification (EVM)** reveals temporal changes in a video that are
invisible to the naked eye: a person's **pulse** (sub-pixel skin-color shifts
with each heartbeat) or a baby's **breathing** (sub-millimeter chest movement).

Instead of tracking features through frames (Lagrangian), EVM treats every pixel
as a time series, amplifies the frequencies of interest, and reconstructs the
video. This notebook builds the CUDA-accelerated pipeline from source and
**profiles every stage** of two pipelines in both FP32 and FP16.

| Pipeline | What it reveals | Decomposition | Temporal filter |
|----------|-----------------|---------------|-----------------|
| **Color** (pulse) | blood flow → green tint | Gaussian downsample (×4) | ideal bandpass (FFT) |
| **Motion** (breathing) | chest movement | Laplacian pyramid (×9) | recursive IIR bandpass |

**Requirements:** a Colab GPU runtime (T4 or better — *Runtime → Change runtime
type → T4 GPU*). The full technical writeup is in the
[repository blog post](https://github.com/iamkucuk/eulerian-video-magnification-cuda/blob/main/docs/blog_speedup.md).

## 1 · Setup

Install build tools + the runtime deps, clone the repo, and compile the CUDA
extension for this GPU's compute capability (a single-arch build is fastest).
`av` (PyAV) encodes the output as H.264 so videos play inline in the browser.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader
!pip install -q cmake ninja pybind11 numpy scipy opencv-python requests av

In [ ]:
import os, shutil, subprocess

REPO = 'https://github.com/iamkucuk/eulerian-video-magnification-cuda.git'
if os.path.exists('evm_cuda'):
    shutil.rmtree('evm_cuda')          # fresh clone picks up the latest fix
subprocess.run(['git', 'clone', REPO, 'evm_cuda'], check=True)
os.chdir('evm_cuda')
print('HEAD:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                              capture_output=True, text=True).stdout.strip())

In [ ]:
import os, shutil, subprocess

# Single-arch build for the attached GPU — fastest compile.
sm = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().replace('.', '')
os.environ['pybind11_DIR'] = subprocess.run(
    ['python3', '-c', 'import pybind11; print(pybind11.get_cmake_dir())'],
    capture_output=True, text=True).stdout.strip()
build = 'cuda/build'
if os.path.exists(build): shutil.rmtree(build)
subprocess.run(['cmake', '-S', 'cuda', '-B', build, '-DCMAKE_BUILD_TYPE=Release',
                f'-DCMAKE_CUDA_ARCHITECTURES={sm}', '-G', 'Ninja'], check=True)
subprocess.run(['cmake', '--build', build, '--config', 'Release', '-j'], check=True)
print(f'Built for sm_{sm}.')

In [ ]:
# Smoke test: the compiled extension loads and sees the GPU.
import sys; sys.path.insert(0, 'cuda')
from evm_cuda import benchmark
print(f'GPU detected: {benchmark.gpu_name()}')
print(f'Free VRAM:    {benchmark.gpu_free_bytes()/1e9:.1f} GB')

In [ ]:
# Fetch the MIT CSAIL sample clips used throughout EVM literature.
subprocess.run(['python', 'scripts/download_samples.py', 'face', 'baby'], check=True)
for f in ['data/face.mp4', 'data/baby.mp4']:
    print(f'  {f}: {os.path.getsize(f)/1e6:.1f} MB')

## 2 · Color magnification — revealing a pulse

Every heartbeat forces blood through the face, causing a sub-pixel shift in
skin color that the eye cannot detect. The color pipeline amplifies it:

1. **color_cvt** — convert BGR→NTSC (YIQ) so luminance and chrominance separate.
2. **blur_dn** — build a 4-level Gaussian pyramid, isolating low-spatial-freq skin.
3. **ideal_bandpass** — keep only the 50–60 bpm band (a person's pulse).
4. **render** — amplify ×50, upsample back, add to the original, quantize to uint8.

The amplified green tint below is blood flow made visible, beat by beat.

In [ ]:
from evm_cuda import benchmark

COLOR = dict(alpha=50, level=4, fl=50/60, fh=60/60,
             chrom_attenuation=1.0, sampling_rate=30.0)

color_fp32 = benchmark.run('color', 'fp32',
                           dict(vid='data/face.mp4', **COLOR),
                           out_path='output/face_fp32.mp4')
print(color_fp32)

In [ ]:
from evm_cuda.colab_utils import show_video   # tiny helper in the repo
show_video('output/face_fp32.mp4', 'Color FP32 — amplified pulse')

## 3 · Motion magnification — revealing breathing

Motion needs a different decomposition. A baby's chest rises and falls by less
than a pixel, so the pipeline builds a **9-level Laplacian pyramid** to isolate
the spatial scale of the movement, then filters each level over time:

- **NTSC** — separate luminance/chrominance (as above).
- **lpyr_build** — 9-level Laplacian pyramid (each level a different spatial scale).
- **IIR** — a recursive bandpass per spatial location (0.4/0.05 cutoffs).
- **render** — reconstruct with the Figure-6 amplification schedule, add back.

> **Note:** FP32 motion needs ~23 GB VRAM. On a 16 GB T4 it is *skipped*
> gracefully (never a crash) — FP16 fits in ~12 GB and runs everywhere.

In [ ]:
MOTION = dict(alpha=10, lambda_c=16, r1=0.4, r2=0.05, chrom_attenuation=0.1)

motion_fp32 = benchmark.run('motion', 'fp32',
                            dict(vid='data/baby.mp4', **MOTION),
                            out_path='output/baby_fp32.mp4')
print(motion_fp32)

In [ ]:
show_video('output/baby_fp32.mp4', 'Motion FP32 — amplified breathing')

## 4 · Why FP16? — halving the bandwidth

The render stage is **memory-bound**: it spends most of its time reading the
NTSC frame, not computing. Storing intermediates as `__half` (2 bytes vs 4)
halves that traffic, so FP16 render is faster on bandwidth-limited GPUs.

The tradeoff is GPU-dependent: on the high-bandwidth A100 the FP32→FP16
conversion overhead can outweigh the savings, but on a 16 GB T4/P100 FP16
both runs **and fits** where FP32 motion would OOM. We reuse the exact same
`benchmark.run` API — only the precision string changes.

In [ ]:
color_fp16 = benchmark.run('color', 'fp16',
                           dict(vid='data/face.mp4', **COLOR),
                           out_path='output/face_fp16.mp4')
motion_fp16 = benchmark.run('motion', 'fp16',
                            dict(vid='data/baby.mp4', **MOTION),
                            out_path='output/baby_fp16.mp4')
print(color_fp16); print(); print(motion_fp16)

## 5 · Fair comparison

All four configurations were measured identically: one warmup run (excluded),
five timed runs, median reported, with a `cudaDeviceSynchronize` after every
stage. Video decode/encode is excluded — these are GPU-pipeline-only numbers.

In [ ]:
print(benchmark.summarize(
    [color_fp32, color_fp16, motion_fp32, motion_fp16], n_iter=5))

## 6 · Output videos

All four magnified clips, encoded as H.264 (playable in the browser):

In [ ]:
show_video('output/face_fp32.mp4', 'Color FP32 (pulse)')
show_video('output/face_fp16.mp4', 'Color FP16 (pulse)')
show_video('output/baby_fp32.mp4', 'Motion FP32 (breathing)')
show_video('output/baby_fp16.mp4', 'Motion FP16 (breathing)')